## Module 3-1 Accessing WRDS with Python

### 0. Installation and Setup
```bash
uv add wrds
```
For documentation, see: https://github.com/wharton/wrds

Offical tutorial: https://wrds-www.wharton.upenn.edu/pages/support/programming-wrds/programming-python/querying-wrds-data-python/

In [ ]:
import wrds
import pandas as pd

### 1. Connecting to WRDS Python Module

In [ ]:
db = wrds.Connection(wrds_username='leonardl')

In [ ]:
# Create a .pgpass file for password authentication
db.create_pgpass_file()

### 2. Querying the Dataset Structure (Metadata)

#### 2.1. List the libraries available at WRDS

In [ ]:
db.list_libraries()

In [ ]:
'crsp' in db.list_libraries()

#### 2.2. List the datasets (tables) in a given Library 

In [ ]:
db.list_tables(library="comp")

In [ ]:
'funda'in db.list_tables(library="comp")

In [ ]:
db.list_tables(library='crsp').index('msf')

In [ ]:
db.list_tables(library='crsp')[229]

In [ ]:
#For example, we can list all tables of Option Price in the OptionMetrics library
[i for i in db.list_tables(library='optionm') if 'opprcd' in i]

### 2.3. List the column headers (variables) within a given dataset:

In [ ]:
db.describe_table(library="crsp", table="dsf_v2")

### 3. Querying WRDS Data Using `raw_sql()`
Executes a SQL query against the specified library and dataset, allowing for highly-granular data queries.

parameters:

- sql - the SQL string to query
- date_cols - a list or dict of column names to parse as date (optional)

In [ ]:
comp = db.raw_sql(
        '''
        SELECT gvkey, datadate, at 
        FROM comp.funda 
            WHERE fyear = 2020 
                AND at IS NOT NULL
        ''', 
        date_cols=['datadate'])
comp.head()

In [ ]:
start_year = 2010
end_year = 2015
query = f"""
        SELECT compustat.*, ccm.permno 
		FROM 
			(
            SELECT gvkey, datadate, fyear, at, ceq, sale, ni 
				FROM comp.funda 
				WHERE fyear BETWEEN {start_year} AND {end_year}
                	AND fyear IS NOT NULL 
					AND indfmt='INDL' AND datafmt='STD' AND popsrc='D' AND consol='C' 
					AND at IS NOT NULL 
                    AND at > 0 
            ) AS compustat, 
            (
            SELECT gvkey, lpermno AS permno, linkdt, linkenddt
				FROM crsp.ccmxpf_linktable
				WHERE linktype in ('LU','LC','LS')
            ) AS ccm
		WHERE compustat.gvkey = ccm.gvkey
			AND (compustat.datadate >= ccm.linkdt OR ccm.linkdt IS NULL)
			AND (compustat.datadate <= ccm.linkenddt OR ccm.linkenddt IS NULL)
		
        ORDER BY compustat.gvkey, fyear, datadate DESC, permno
        """

In [ ]:
with wrds.Connection(wrds_username='leonardl') as db:
    CCM = (
        db.raw_sql(query, date_cols=['datadate'])
        .drop_duplicates(subset=['gvkey', 'fyear'])
        )
CCM.head()

### 4. Save the file to Stata file or csv file

In [ ]:
CCM.to_stata('Comp_with_permno.dta', write_index=False, convert_dates={'datadate': 'td'})
# CCM.to_csv('Comp_with_permno.csv', sep='\t', index=False)